In [ ]:
# подключение Google Диска
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# импорт библиотек
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
# путь к папке с файлами
base_path = "/content/drive/MyDrive/Series/"

In [ ]:
# загрузка данных
female_births = pd.read_csv(base_path + "daily-total-female-births-in-cal.csv", parse_dates=["Date"])
airline = pd.read_csv(base_path + "international-airline-passengers.csv", parse_dates=["Month"])

In [ ]:
# берём числовые ряды
series_stationary = female_births["Count"]          # стационарный (примерно)
series_nonstationary = airline["Count"]             # нестационарный (тренд + сезонность)

In [ ]:
# модели скользящего среднего
def moving_average(series, window):
    return series.rolling(window).mean()

def exponential_moving_average(series, span):
    return series.ewm(span=span, adjust=False).mean()

def dema(series, span):
    ema1 = exponential_moving_average(series, span)
    ema2 = exponential_moving_average(ema1, span)
    return 2 * ema1 - ema2

def tema(series, span):
    ema1 = exponential_moving_average(series, span)
    ema2 = exponential_moving_average(ema1, span)
    ema3 = exponential_moving_average(ema2, span)
    return 3 * ema1 - 3 * ema2 + ema3

models = {
    'MA': moving_average,
    'EMA': exponential_moving_average,
    'DEMA': dema,
    'TEMA': tema
}

In [ ]:
# функция подбора оптимального параметра
def find_best_param(train, test_point, model_func, param_range, is_ema=False):
    best_param = None
    best_mae = float('inf')
    best_metrics = {}
    for p in param_range:
        if is_ema:
            fitted = model_func(train, span=p)
        else:
            fitted = model_func(train, window=p)
        if fitted.isna().all():
            continue
        forecast = fitted.iloc[-1]
        mae = mean_absolute_error([test_point], [forecast])
        if mae < best_mae:
            best_mae = mae
            best_param = p
            best_metrics = {
                'MAE': mae,
                'MSE': mean_squared_error([test_point], [forecast]),
                'RMSE': np.sqrt(mean_squared_error([test_point], [forecast]))
            }
    return best_param, best_metrics

In [ ]:
# эксперимент для одного ряда
def run_experiment(series, param_range=range(2, 31)):
    train = series.iloc[:-1]          # все кроме последней точки
    test_point = series.iloc[-1]      # последняя точка - тест
    results = {}
    for name, func in models.items():
        is_ema = name in ['EMA', 'DEMA', 'TEMA']
        best_param, metrics = find_best_param(train, test_point, func, param_range, is_ema)
        # прогноз с лучшим параметром
        forecast = None
        if best_param is not None:
            if is_ema:
                fitted = func(train, span=best_param)
            else:
                fitted = func(train, window=best_param)
            if not fitted.isna().all():
                forecast = fitted.iloc[-1]
        results[name] = {
            'best_param': best_param,
            'MAE': metrics.get('MAE'),
            'MSE': metrics.get('MSE'),
            'RMSE': metrics.get('RMSE'),
            'forecast': forecast
        }
    return results, test_point

In [ ]:
# Запуск
print("="*60)
print("СТАЦИОНАРНЫЙ РЯД (ежедневные рождения)")
print("="*60)
res_stat, actual_stat = run_experiment(series_stationary)
for model, vals in res_stat.items():
    print(f"{model:6} | param={vals['best_param']:2} | MAE={vals['MAE']:.4f} | RMSE={vals['RMSE']:.4f} | forecast={vals['forecast']:.2f} (actual={actual_stat})")

print("\n" + "="*60)
print("НЕСТАЦИОНАРНЫЙ РЯД (авиапассажиры)")
print("="*60)
res_nonstat, actual_nonstat = run_experiment(series_nonstationary)
for model, vals in res_nonstat.items():
    print(f"{model:6} | param={vals['best_param']:2} | MAE={vals['MAE']:.4f} | RMSE={vals['RMSE']:.4f} | forecast={vals['forecast']:.2f} (actual={actual_nonstat})")

СТАЦИОНАРНЫЙ РЯД (ежедневные рождения)
MA     | param= 2 | MAE=1.5000 | RMSE=1.5000 | forecast=51.50 (actual=50)
EMA    | param= 3 | MAE=0.6846 | RMSE=0.6846 | forecast=50.68 (actual=50)
DEMA   | param= 7 | MAE=0.0129 | RMSE=0.0129 | forecast=50.01 (actual=50)
TEMA   | param=11 | MAE=0.2402 | RMSE=0.2402 | forecast=49.76 (actual=50)

НЕСТАЦИОНАРНЫЙ РЯД (авиапассажиры)
MA     | param=27 | MAE=6.2222 | RMSE=6.2222 | forecast=438.22 (actual=432)
EMA    | param= 2 | MAE=9.7409 | RMSE=9.7409 | forecast=422.26 (actual=432)
DEMA   | param= 4 | MAE=5.3162 | RMSE=5.3162 | forecast=426.68 (actual=432)
TEMA   | param= 6 | MAE=4.2058 | RMSE=4.2058 | forecast=427.79 (actual=432)


In [ ]:
# Краткий вывод
print("\n" + "="*60)
print("ВЫВОДЫ")
print("="*60)
# Определяем лучшую модель по RMSE (можно и по MAE)
best_stat = min(res_stat, key=lambda x: res_stat[x]['RMSE'] if res_stat[x]['RMSE'] is not None else float('inf'))
best_nonstat = min(res_nonstat, key=lambda x: res_nonstat[x]['RMSE'] if res_nonstat[x]['RMSE'] is not None else float('inf'))

print(f"Для стационарного ряда лучшая модель: {best_stat} (RMSE={res_stat[best_stat]['RMSE']:.4f})")
print(f"Для нестационарного ряда лучшая модель: {best_nonstat} (RMSE={res_nonstat[best_nonstat]['RMSE']:.4f})")
print("\nОбоснование:")
print("- Стационарный ряд не имеет тренда, поэтому простое скользящее среднее (MA) даёт стабильный прогноз.")
print("- Нестационарный ряд имеет тренд и сезонность; TEMA быстрее реагирует на изменения, поэтому даёт меньшую ошибку.")


ВЫВОДЫ
Для стационарного ряда лучшая модель: DEMA (RMSE=0.0129)
Для нестационарного ряда лучшая модель: TEMA (RMSE=4.2058)

Обоснование:
- Стационарный ряд не имеет тренда, поэтому простое скользящее среднее (MA) даёт стабильный прогноз.
- Нестационарный ряд имеет тренд и сезонность; TEMA быстрее реагирует на изменения, поэтому даёт меньшую ошибку.
